In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
%load_ext autoreload
%autoreload 2

In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
import yfinance as yf
import statsmodels.api as sm
import data as dt
import models as md
import backtest as bt
import run as rn
from pathlib import Path
print("OK")

output_dir = Path("../Outputs")

residuals = pd.read_csv(output_dir / "residuals.csv",
                        index_col=0, parse_dates = True)
print(f"Residuals loaded: {residuals.shape}")
print(f"Tickers: {residuals.columns.tolist()}")

OK
Residuals loaded: (1096, 7)
Tickers: ['DAL', 'UAL', 'AAL', 'LUV', 'ALK', 'JBLU', 'CPA']


In [3]:
univ = ['DAL','UAL','AAL','JBLU','LUV','ALK','CPA','JETS','BZ=F','^VIX','SPY']
px = yf.download(univ, start='2016-01-01', auto_adjust=True,
                   progress=False)['Close']
px = px.rename(columns={'BZ=F':'Brent', '^VIX':'VIX'})
ret = px.pct_change().dropna()
print(f"Downloaded: {px.shape}  |  {px.index[0].date()} → {px.index[-1].date()}")

Downloaded: (2663, 11)  |  2016-01-04 → 2026-07-31


In [19]:
output_dir = Path("../Outputs")
ticker = 'DAL'
zwin = 45
start = '2022-01-01'
end = '2026-07-29'
residuals = pd.read_csv(output_dir / "residuals.csv", index_col=0, parse_dates=True)
prices = pd.read_csv(output_dir / "prices.csv", index_col=0, parse_dates=True)

print(f"Residuals: {residuals.shape} | {residuals.index[0].date()} → {residuals.index[-1].date()}")
print(f"Prices: {prices.shape} | {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"Available tickers: {residuals.columns.tolist()}")

Residuals: (1096, 7) | 2022-03-09 → 2026-07-16
Prices: (2904, 11) | 2015-01-02 → 2026-07-16
Available tickers: ['DAL', 'UAL', 'AAL', 'LUV', 'ALK', 'JBLU', 'CPA']


In [20]:
zscores_full = md.compute_rolling_zscore(residuals, window=zwin)

if ticker not in residuals.columns:
    print(f"ERROR: {ticker} not in residuals. Available: {residuals.columns.tolist()}")
else:

    price_s = prices[ticker].dropna()
    z_s = zscores_full[ticker].dropna()

    common = price_s.index.intersection(z_s.index)
    common = common[(common >= start) & (common <= end)]

    print(f"Common dates in range: {len(common)}")
    if len(common) == 0:
        print(f"No overlap. Price range: {price_s.index[0].date()} → {price_s.index[-1].date()}")
        print(f"Z-score range: {z_s.index[0].date()} → {z_s.index[-1].date()}")
    else:
        price_plot = price_s.loc[common]
        z_plot = z_s.loc[common]
        print(f"Price range: {price_plot.min():.2f} → {price_plot.max():.2f}")
        print(f"Z-score range: {z_plot.min():.3f} → {z_plot.max():.3f}")
        print(f"Signals (z>+2): {(z_plot > 2).sum()} (z<-2): {(z_plot < -2).sum()}")

Common dates in range: 1083
Price range: 27.05 → 93.43
Z-score range: -4.579 → 4.000
Signals (z>+2): 33 (z<-2): 30


In [21]:
if len(common) > 0:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8),
                                    sharex=True,
                                    gridspec_kw={'height_ratios': [2, 1]})

    ax1.plot(price_plot.index, price_plot.values,
             color='navy', linewidth=1.1, label=f'{ticker} Price')
    ax1.plot(price_plot.rolling(50).mean().index,
             price_plot.rolling(50).mean().values,
             color='orange', linewidth=1.0, linestyle='--',
             alpha=0.8, label='50d MA')

    for date in common:
        z_val = z_plot.loc[date]
        color = 'green' if z_val > 2.0 else ('red' if z_val < -2.0 else None)
        if color:
            ax1.axvspan(date, date + pd.Timedelta(days=1),
                        color=color, alpha=0.07)

    ax1.set_ylabel("Price ($)", fontsize=11)
    ax1.set_title(f"{ticker} | Window={zwin}d  |  {start} → {end}\n"
                  f"Green = z>+2 (momentum long / mean-rev short)  "
                  f"Red = z<-2 (momentum short / mean-rev long)")
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(alpha=0.3)

    ax2.plot(z_plot.index, z_plot.values,
             color='steelblue', linewidth=0.8, label='Z-Score')
    ax2.fill_between(z_plot.index, z_plot.values, 0,
                     where=(z_plot > 0), color='green', alpha=0.08)
    ax2.fill_between(z_plot.index, z_plot.values, 0,
                     where=(z_plot < 0), color='red', alpha=0.08)

    for level, color, style in [(2.0, 'green', '--'),
                                 (-2.0, 'red',   '--'),
                                 (bt.exit_threshold,  'gray', ':'),
                                 (-bt.exit_threshold, 'gray', ':'),
                                 (0.0, 'black', '-')]:
        ax2.axhline(level, color=color, linestyle=style, linewidth=0.8)

    ax2.set_ylim(-4.5, 4.5)
    ax2.set_ylabel("Z-Score", fontsize=11)
    ax2.set_xlabel("Date", fontsize=10)
    ax2.legend(['Z-Score'], loc='upper left', fontsize=9)
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ── Key question ──────────────────────────────────────────────────────────
    print(f"\nKey question — look at green shaded regions (z>+2):")
    print(f"Do they coincide with price RISING?  → momentum mode correct")
    print(f"Do they coincide with price FALLING? → mean reversion correct")


Key question — look at green shaded regions (z>+2):
Do they coincide with price RISING?  → momentum mode correct
Do they coincide with price FALLING? → mean reversion correct


In [22]:
airlines = ['DAL','UAL','AAL','LUV','ALK','JBLU','CPA']
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
axes = axes.flatten()
for i, t in enumerate(airlines):
    p = px[t].dropna()
    p0 = p.iloc[0]
    axes[i].plot(p.index, p/p0, color='steelblue', linewidth=1.2, label='Price')
    axes[i].plot((p.rolling(50).mean()/p0).index,
                  p.rolling(50).mean()/p0,
                  color='red', linewidth=1.0, linestyle='--', label='50d MA')
    axes[i].plot((p.rolling(100).mean()/p0).index,
                  p.rolling(100).mean()/p0,
                  color='orange', linewidth=1.0, linestyle='--', label='100d MA')
    axes[i].plot((p.rolling(200).mean()/p0).index,
                  p.rolling(200).mean()/p0,
                  color='green', linewidth=1.0, linestyle='--', label='200d MA')
    axes[i].set_title(t)
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel("Normalized Price")
if len(airlines) < len(axes):
    axes[-1].set_visible(False)
plt.suptitle("Airline Price Levels with 50/100/200-Day Moving Averages", fontsize=16)
plt.tight_layout()
plt.show()

In [23]:
spy_ret = ret['SPY']
corr_r = ret[airlines].rolling(252).corr(spy_ret)
vol_r = ret[airlines].rolling(252).std()
vol_spy = ret['SPY'].rolling(252).std()
beta = corr_r.multiply(vol_r, axis=0).divide(vol_spy, axis=0)
resid = ret[airlines].sub(beta.multiply(spy_ret, axis=0), axis=0)
fig, axes = plt.subplots(4, 2, figsize=(16, 12))
axes = axes.flatten()
for i, t in enumerate(airlines):
    r = resid[t].dropna()
    axes[i].plot(r.cumsum(), color='steelblue', linewidth=1.0)
    axes[i].axhline(0, color='black', linewidth=0.6)
    axes[i].set_title(f"{t} Cumulative SPY-Adjusted Residual")
    axes[i].set_ylabel("Cumsum(residual)")
if len(airlines) < len(axes):
    axes[-1].set_visible(False)
plt.suptitle("Point-in-Time Residuals vs SPY Beta", fontsize=13)
plt.tight_layout()
plt.show()

In [24]:
df_stats = pd.DataFrame({
    'IR': (resid[airlines].mean() / resid[airlines].std() * np.sqrt(252)),
    'SR': (ret[airlines].mean() / ret[airlines].std() * np.sqrt(252)),
})
print("Information Ratio (residual) vs. Sharpe Ratio (raw):")
print(df_stats.round(3).to_string())

df_stats.plot(kind='bar', figsize=(10, 5),
    title="IR (residual/SPY) vs Sharpe (raw) — higher IR = more idiosyncratic alpha")
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=0)
plt.show()

Information Ratio (residual) vs. Sharpe Ratio (raw):
           IR     SR
Ticker              
DAL    -0.250  0.339
UAL    -0.157  0.376
AAL    -0.499  0.060
LUV    -0.372  0.208
ALK    -0.540  0.093
JBLU   -0.444  0.031
CPA    -0.011  0.526


In [25]:
vol_compare = pd.DataFrame({
    'Raw Vol': ret[airlines].std() * np.sqrt(252),
    'Residual Vol': resid[airlines].std() * np.sqrt(252),
})
print("Volatility Comparison:")
print(vol_compare.round(4))

vol_compare.plot(kind='bar', figsize=(10,5),
    title="Raw vs SPY-Adjusted Residual Volatility (annualized)")
plt.xticks(rotation=0)
plt.ylabel("Annualized Volatility")
plt.show()

Volatility Comparison:
        Raw Vol  Residual Vol
Ticker                       
DAL      0.4163        0.3419
UAL      0.5121        0.4375
AAL      0.5237        0.4609
LUV      0.3735        0.3211
ALK      0.4323        0.3636
JBLU     0.5415        0.4941
CPA      0.4500        0.3922


In [26]:
fig, axes = plt.subplots(4, 2, figsize=(16, 12))
axes = axes.flatten()
for i, t in enumerate(airlines):
    others = [x for x in airlines if x != t]
    rolling_corr = ret[t].rolling(90).corr(ret[others].mean(axis=1))
    axes[i].plot(rolling_corr, color='purple', linewidth=0.8)
    axes[i].axhline(rolling_corr.mean(), color='red', linestyle='--',
                    linewidth=0.8, label=f"Mean: {rolling_corr.mean():.2f}")
    axes[i].set_title(f"{t} — Rolling 90d Corr with Airline Peers")
    axes[i].set_ylim(-0.2, 1.0)
    axes[i].legend(fontsize=8)
if len(airlines) < len(axes):
    axes[-1].set_visible(False)
plt.suptitle("Rolling Correlation With Airline Peer Group", fontsize=13)
plt.tight_layout()
plt.show()

In [27]:
fig, axes = plt.subplots(4, 2, figsize=(15, 9))
axes = axes.flatten()
brent_monthly = ret['Brent'].resample('ME').sum()
for i, t in enumerate(airlines):
    air_monthly = ret[t].resample('ME').sum()
    common = brent_monthly.index.intersection(air_monthly.index)
    x = brent_monthly.loc[common]
    y = air_monthly.loc[common]
    axes[i].scatter(x, y, alpha=0.5, s=20, color='steelblue')
    m, b_, *_ = np.polyfit(x, y, 1)
    xl = np.linspace(x.min(), x.max(), 100)
    axes[i].plot(xl, m*xl + b_, color='red', linewidth=1.0)
    corr = x.corr(y)
    axes[i].set_title(f"{t}  (Brent β: {m:.2f}  ρ: {corr:.2f})")
    axes[i].set_xlabel("Brent Monthly Return")
    axes[i].set_ylabel(f"{t} Monthly Return")
if len(airlines) < len(axes):
    axes[-1].set_visible(False)
plt.suptitle("Brent Crude vs Airline Monthly Returns — Sensitivity Analysis",
             fontsize=12)
plt.tight_layout()
plt.show()

In [28]:
fig, axes = plt.subplots(4, 2, figsize=(15, 9))
axes = axes.flatten()
vix_monthly = ret['VIX'].resample('ME').sum()
for i, t in enumerate(airlines):
    air_monthly = ret[t].resample('ME').sum()
    common = vix_monthly.index.intersection(air_monthly.index)
    x = vix_monthly.loc[common]
    y = air_monthly.loc[common]
    axes[i].scatter(x, y, alpha=0.5, s=20, color='steelblue')
    m, b_, *_ = np.polyfit(x, y, 1)
    xl = np.linspace(x.min(), x.max(), 100)
    axes[i].plot(xl, m*xl + b_, color='red', linewidth=1.0)
    corr = x.corr(y)
    axes[i].set_title(f"{t}  (VIX β: {m:.2f}  ρ: {corr:.2f})")
    axes[i].set_xlabel("VIX Monthly Return")
    axes[i].set_ylabel(f"{t} Monthly Return")
if len(airlines) < len(axes):
    axes[-1].set_visible(False)
plt.suptitle("VIX vs Airline Monthly Returns — Sensitivity Analysis",
             fontsize=12)
plt.tight_layout()
plt.show()

In [29]:
fig, axes = plt.subplots(4, 2, figsize=(15, 9))
axes = axes.flatten()
jets_monthly = ret['JETS'].resample('ME').sum()
for i, t in enumerate(airlines):
    air_monthly = ret[t].resample('ME').sum()
    common = jets_monthly.index.intersection(air_monthly.index)
    x = jets_monthly.loc[common]
    y = air_monthly.loc[common]
    axes[i].scatter(x, y, alpha=0.5, s=20, color='steelblue')
    m, b_, *_ = np.polyfit(x, y, 1)
    xl = np.linspace(x.min(), x.max(), 100)
    axes[i].plot(xl, m*xl + b_, color='red', linewidth=1.0)
    corr = x.corr(y)
    axes[i].set_title(f"{t}  (JETS β: {m:.2f}  ρ: {corr:.2f})")
    axes[i].set_xlabel("JETS Monthly Return")
    axes[i].set_ylabel(f"{t} Monthly Return")
if len(airlines) < len(axes):
    axes[-1].set_visible(False)
plt.suptitle("JETS vs Airline Monthly Returns — Sensitivity Analysis",
             fontsize=12)
plt.tight_layout()
plt.show()

In [30]:
output_dir = Path("../Outputs")

airline_returns = pd.read_csv(
    output_dir / "airline_returns.csv", index_col=0, parse_dates=True
)
macro_returns = pd.read_csv(
    output_dir / "residuals.csv", index_col=0, parse_dates=True
)
residuals = pd.read_csv(
    output_dir / "residuals.csv", index_col=0, parse_dates=True
)
print(f"airline_returns: {airline_returns.shape}")
print(f"residuals: {residuals.shape}")

airline_returns: (2904, 11)
residuals: (1096, 7)


In [31]:
windows = [20, 30, 45, 60, 75, 90]
for w in windows:
    z = md.compute_rolling_zscore(residuals, window=w)
    signals = bt.generate_signals(z)
    positions = bt.construct_portfolio(signals)
    net, trades = bt.compute_portfolio_returns(
        positions,
        airline_returns
    )
    perf = bt.compute_performance(
        net,
        trades,
        macro_returns
    )

    print(f"\nWindow = {w}")
    print(perf[["annualized_returns", "sharpe_ratio", "ann_turnover"]])


Window = 20
annualized_returns   -660.2219
sharpe_ratio             -2.05
ann_turnover            45.016
dtype: object

Window = 30
annualized_returns   -486.653
sharpe_ratio           -1.474
ann_turnover           42.372
dtype: object

Window = 45
annualized_returns    198.7716
sharpe_ratio             0.627
ann_turnover            40.774
dtype: object

Window = 60
annualized_returns    127.0926
sharpe_ratio             0.412
ann_turnover            39.923
dtype: object

Window = 75
annualized_returns    22.5501
sharpe_ratio            0.074
ann_turnover           40.222
dtype: object

Window = 90
annualized_returns   -372.4055
sharpe_ratio            -1.175
ann_turnover            37.953
dtype: object
